In [3]:
# --- Imports ---
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.signal import butter, filtfilt, resample, find_peaks
import neurokit2 as nk
from scipy.stats import wilcoxon

# --- Pandas display settings (optional, for debugging) ---
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 1000)

print("✅ Imports loaded")


✅ Imports loaded


In [4]:
# --- Respiration file paths (.h5) ---
resp_paths = {
    "RI1_3_6": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s3_6_p5_3_nRB3_20250621_125312_merged.h5",
    "RI2_3_6": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s3_6_p5_3_nRB3_20250621_131158_merged.h5",
    "RI1_4_7": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s4_7_p5_2_nRB3_20250621_150707_merged.h5",
    "RI2_4_7": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s4_7_p5_2_nRB3_20250621_152519_merged.h5",
    "RI1_2_3": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s2_3_p5_3_nRB3_20250622_104059_merged (1).h5",
    "RI2_2_3": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s2_3_p5_3_nRB3_20250622_110216_merged.h5",
    "RI1_4_8": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s4_8_p5_1_nRB3_20250621_165214_merged.h5",
    "RI2_4_8": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s4_8_p5_1_nRB3_20250621_171318_merged.h5",
    "RI1_1_1": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s1_1_p5_2_nRB6_20250622_143958_merged.h5",
    "RI2_1_1": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s1_1_p5_2_nRB6_20250622_150457_merged.h5",
    "RI1_1_2": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s1_2_p5_1_nRB6_20250622_170742_merged.h5",
    "RI2_1_2": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s1_2_p5_1_nRB6_20250622_173049_merged.h5",
    "RI1_2_4": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s2_4_p5_4_nRB3_20250622_123424_merged.h5",
    "RI2_2_4": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s2_4_p5_4_nRB3_20250622_125648_merged.h5",
    "RI1_3_5": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s3_5_p5_4_nRB3_20250621_105014_merged.h5",
    "RI2_3_5": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s3_5_p5_4_nRB3_20250621_112618_merged.h5",
    # Baseline recordings
    "BLRI_1_1": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s1_1_p5_2_nRB6_20250622_141846_merged.h5",
    "BLRI_1_2": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s1_2_p5_1_nRB6_20250622_164833_merged.h5",
    "BLRI_2_3": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s2_3_p5_3_nRB3_20250622_101813_merged.h5",
    "BLRI_2_4": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s2_4_p5_4_nRB3_20250622_121708_merged.h5",
    "BLRI_3_6": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s3_6_p5_3_nRB3_20250621_123634_merged.h5",
    "BLRI_4_7": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s4_7_p5_2_nRB3_20250621_144806_merged.h5",
    "BLRI_4_8": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s4_8_p5_1_nRB3_20250621_163506_merged.h5",
}

# --- BORIS annotation file paths (.csv) ---
boris_paths = {
    "RI1_3_6": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s3_6_p5_3_nRB3_HEEPS.csv",
    "RI2_3_6": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s3_6_p_5_3_nRB3_2025062.csv",
    "RI1_4_7": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s4_7_p5_2_nRB3_HEEPS.csv",
    "RI2_4_7": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s4_7_p5_2_nRB3_HEEPS.csv",
    "RI1_2_3": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s2_3_p5_3_nRB3_20250622_104059.1.csv",
    "RI2_2_3": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s2_3_p5_3_nRB3_20250622_1102116.1.csv",
    "RI1_4_8": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s4_8_p5_1_nRB3_HEEPS.csv",
    "RI2_4_8": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s4_8_p5_1_nRB3_20250621_HEEPS.csv",
    "RI1_1_1": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s1_1_p5_2_nRB6_20250622_143958.1.csv",
    "RI2_1_1": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s1_1_p5_2_nRB6_20250622_150457.1.csv",
    "RI1_1_2": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s1_2_p5_1_nRB6_20250622_170742.1.csv",
    "RI2_1_2": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s1_2_p5_1_nRB6_20250622_173049.1.csv",
    "RI1_2_4": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s2_4_p5_4_nRB3_20250622_123424.1.csv",
    "RI2_2_4": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s2_4_p5_4_nRB3_20250622_125648.1.csv",
    "RI2_3_5": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s3_5_p5_4_nRB3_20250621.csv",
}

print(f"✅ Resp files: {len(resp_paths)} trials")
print(f"✅ BORIS files: {len(boris_paths)} trials")


✅ Resp files: 23 trials
✅ BORIS files: 15 trials


In [5]:
# --- Load BORIS CSVs into DataFrames (auto from boris_paths) ---
boris_data = {}

for label, path in boris_paths.items():
    try:
        df = pd.read_csv(path)
        boris_data[label] = df.copy()
        print(f"Loaded BORIS: {label} ({df.shape[0]} rows)")
    except Exception as e:
        print(f"❌ Error loading {label}: {e}")

print(f"✅ Loaded {len(boris_data)} BORIS files")


Loaded BORIS: RI1_3_6 (37 rows)
Loaded BORIS: RI2_3_6 (109 rows)
Loaded BORIS: RI1_4_7 (37 rows)
Loaded BORIS: RI2_4_7 (108 rows)
Loaded BORIS: RI1_2_3 (29 rows)
Loaded BORIS: RI2_2_3 (82 rows)
Loaded BORIS: RI1_4_8 (55 rows)
Loaded BORIS: RI2_4_8 (113 rows)
Loaded BORIS: RI1_1_1 (92 rows)
Loaded BORIS: RI2_1_1 (37 rows)
Loaded BORIS: RI1_1_2 (89 rows)
Loaded BORIS: RI2_1_2 (58 rows)
Loaded BORIS: RI1_2_4 (69 rows)
Loaded BORIS: RI2_2_4 (77 rows)
Loaded BORIS: RI2_3_5 (27 rows)
✅ Loaded 15 BORIS files


In [6]:
import h5py

def load_clean_resp_signal(h5_file, target_rate=100):
    """
    Load respiration from .h5, resample to target_rate, band-pass clean.
    Returns: (resp_signal, time_vector, sampling_rate)
    """
    try:
        with h5py.File(h5_file, 'r') as f:
            resp = f['resp'][:].flatten()
            ekg_meta = dict(f['ekg_metadata'].attrs)
        duration_sec = float(ekg_meta['duration_sec'])
        fs = len(resp) / duration_sec
    except Exception as e:
        print(f"❌ Error loading {h5_file}: {e}")
        return None, None, None

    # --- Low-pass filter to remove noise ---
    cutoff_hz = min(0.45 * fs, 20.0) if target_rate >= fs else target_rate / 2.0
    Wn = cutoff_hz / (fs / 2.0)
    b, a = butter(N=4, Wn=Wn, btype='low')
    filtered = filtfilt(b, a, resp)

    # --- Resample to EXACT target_rate ---
    N_out = int(round(duration_sec * target_rate))
    resampled = resample(filtered, N_out)

    # --- Final band-pass clean (0.1–15 Hz) ---
    rsp_cleaned = nk.signal_filter(
        resampled, lowcut=0.1, highcut=15.0,
        method="butterworth", sampling_rate=target_rate, order=2
    )

    time_vector = np.arange(len(rsp_cleaned)) / float(target_rate)
    return rsp_cleaned, time_vector, float(target_rate)


# --- Load all respiration files into a dictionary ---
resp_data = {}
for label, path in resp_paths.items():
    print(f"Processing {label}...")
    signal, time, rate = load_clean_resp_signal(path)
    if signal is not None:
        resp_data[label] = {
            "signal": signal,
            "time": time,
            "sampling_rate": rate
        }

print(f"✅ Loaded {len(resp_data)} respiration traces")


Processing RI1_3_6...


MemoryError: bad allocation

In [23]:
ri1_keys = [k for k in resp_paths if k.startswith("RI1")]
ri2_keys = [k for k in resp_paths if k.startswith("RI2")]
blri_keys = [k for k in resp_paths if k.startswith("BLRI")]
print(f"RI1 trials: {ri1_keys}")


RI1 trials: ['RI1_3_6', 'RI1_4_7', 'RI1_2_3', 'RI1_4_8', 'RI1_1_1', 'RI1_1_2', 'RI1_2_4', 'RI1_3_5']


In [25]:
resp_data["RI1_3_6"]           # respiration trace for RI1_3_6

{'signal': array([ 209.87636112,    8.80604237, -115.251069  , ...,   61.64067254,
          25.27417181,   -5.85655201]),
 'time': array([0.0000e+00, 1.0000e-02, 2.0000e-02, ..., 6.0324e+02, 6.0325e+02,
        6.0326e+02]),
 'sampling_rate': 100.0}

In [ ]:
boris_data["RI1_3_6"].head()   # behavior annotations for RI1_3_6



,Observation id,Observation date,Description,Observation type,Source,Time offset (s),Coding duration,Media duration (s),FPS (frame/s),Subject,Observation duration by subject by observation,Behavior,Behavioral category,Behavior type,Start (s),Stop (s),Duration (s),Media file name,Image index start,Image index stop,Image file path start,Image file path stop,Comment start,Comment stop
0,RI1_s3_6_p5_3_nRB3,2025-06-28 22:13:15.619,NaN,Media file,player #1:C:/Users/thoma/Desktop/BorisVidsObse...,0.0,589.517,622.862,29.0,social_agent,23.313,Anogenital Sniffing,Not defined,STATE,8.862,9.448,0.586,C:/Users/thoma/Desktop/BorisVidsObservation/RI...,257,274.0,NaN,NaN,NaN,NaN
1,RI1_s3_6_p5_3_nRB3,2025-06-28 22:13:15.619,NaN,Media file,player #1:C:/Users/thoma/Desktop/BorisVidsObse...,0.0,589.517,622.862,29.0,subject,22.135,Facial Sniffing,Not defined,STATE,11.966,13.000,1.034,C:/Users/thoma/Desktop/BorisVidsObservation/RI...,347,377.0,NaN,NaN,NaN,NaN
2,RI1_s3_6_p5_3_nRB3,2025-06-28 22:13:15.619,NaN,Media file,player #1:C:/Users/thoma/Desktop/BorisVidsObse...,0.0,589.517,622.862,29.0,social_agent,23.313,Facial Sniffing,Not defined,STATE,12.069,12.862,0.793,C:/Users/thoma/Desktop/BorisVidsObservation/RI...,350,373.0,NaN,NaN,NaN,NaN
3,RI1_s3_6_p5_3_nRB3,2025-06-28 22:13:15.619,NaN,Media file,player #1:C:/Users/thoma/Desktop/BorisVidsObse...,0.0,589.517,622.862,29.0,subject,22.135,Body Sniffing,Not defined,STATE,13.276,13.448,0.172,C:/Users/thoma/Desktop/BorisVidsObservation/RI...,385,390.0,NaN,NaN,NaN,NaN
4,RI1_s3_6_p5_3_nRB3,2025-06-28 22:13:15.619,NaN,Media file,player #1:C:/Users/thoma/Desktop/BorisVidsObse...,0.0,589.517,622.862,29.0,subject,22.135,Anogenital Sniffing,Not defined,STATE,14.483,16.690,2.207,C:/Users/thoma/Desktop/BorisVidsObservation/RI...,420,484.0,NaN,NaN,NaN,NaN


## Step 1: Clean BORIS Data

**Goal:** Make sure all behavior and subject labels are consistent across BORIS files before we do any analysis.

### Why this matters
- BORIS sometimes exports labels with extra spaces, different capitalization, or typos.  
  - Example: `"Facial sniffing "`, `"facial Sniffing"`, and `"facial sniffing"` would be treated as **different behaviors** by the computer.  
- If we don’t clean this, our counts and plots will be wrong because the same behavior gets split into multiple categories.  
- Cleaning makes all behaviors and subject labels standardized.

### What we do in this step
1. Remove extra spaces from labels.  
2. Convert everything to lowercase (so `"Facial sniffing"` → `"facial sniffing"`).  
3. Map known behaviors back to **pretty, standardized names**:
   - `"facial sniffing"` → `"Facial Sniffing"`
   - `"body sniffing"` → `"Body Sniffing"`
   - `"anogenital sniffing"` → `"Anogenital Sniffing"`
   - `"fighting"` → `"Fighting"`
   - `"chasing"` → `"Chasing"`
   - `"tail rattling"` → `"Tail Rattling"`
   - `"posturing"` → `"Posturing"`

### What we get after cleaning
- All BORIS files now use the same labels.  
- Behaviors are easy to group and count correctly.  
- Plots will look nice and readable.  


In [20]:
# --- Standardize behavior + subject labels in BORIS ---
def clean_boris(df):
    df = df.copy()
    # Normalize text
    df['Behavior'] = df['Behavior'].str.strip().str.lower()
    df['Subject'] = df['Subject'].str.strip().str.lower()
    
    # Map to pretty labels (covering all relevant behaviors)
    behavior_label_map = {
        "facial sniffing": "Facial Sniffing",
        "body sniffing": "Body Sniffing",
        "anogenital sniffing": "Anogenital Sniffing",
        "fighting": "Fighting",
        "chasing": "Chasing",
        "tail rattling": "Tail Rattling",  # correct spelling
        "posturing": "Posturing"
    }
    
    # Apply mapping (if not in map, keep original cleaned value)
    df['Behavior'] = df['Behavior'].map(behavior_label_map).fillna(df['Behavior'])
    
    return df

# Apply cleaning to all BORIS files
for key in boris_data:
    boris_data[key] = clean_boris(boris_data[key])


What Step 2 actually does

Loops over boris_data dictionary.

For each file:

Adds "Trial" (so you know which session it was).

Adds "Condition" (Positive if RI1, Negative if RI2, Baseline if BLRI).

Extracts "Subject_ID" (like "3_6" from "RI1_3_6").

Appends all of them together → combined_bouts DataFrame.

In [29]:
# --- Combine all BORIS data ---
all_bouts = []

for trial, df in boris_data.items():
    if trial.startswith("RI1"):
        condition = "Positive"
    elif trial.startswith("RI2"):
        condition = "Negative"
    elif trial.startswith("BLRI"):
        condition = "Baseline"
    else:
        condition = "Other"
    
    temp = df.copy()
    temp["Trial"] = trial
    temp["Condition"] = condition
    
    # Extract subject ID from trial name, e.g. RI1_3_6 → "3_6"
    parts = trial.split("_")
    if len(parts) >= 3:
        temp["Subject_ID"] = parts[1] + "_" + parts[2]
    else:
        temp["Subject_ID"] = "Unknown"
    
    all_bouts.append(temp)

combined_bouts = pd.concat(all_bouts, ignore_index=True)
print("✅ Combined BORIS data:", combined_bouts.shape)
combined_bouts


✅ Combined BORIS data: (1019, 31)


,Observation id,Observation date,Description,Observation type,Source,Time offset (s),Coding duration,Media duration (s),FPS (frame/s),Subject,Observation duration by subject by observation,Behavior,Behavioral category,Behavior type,Start (s),Stop (s),Duration (s),Media file name,Image index start,Image index stop,Image file path start,Image file path stop,Comment start,Comment stop,Trial,Condition,Subject_ID,Media file,Total length,FPS,Modifiers
0,RI1_s3_6_p5_3_nRB3,2025-06-28 22:13:15.619,NaN,Media file,player #1:C:/Users/thoma/Desktop/BorisVidsObse...,0.0,589.517,622.862,29.0,social_agent,23.313,Anogenital Sniffing,Not defined,STATE,8.862,9.448,0.586,C:/Users/thoma/Desktop/BorisVidsObservation/RI...,257.0,274.0,NaN,NaN,NaN,NaN,RI1_3_6,Positive,3_6,NaN,NaN,NaN,NaN
1,RI1_s3_6_p5_3_nRB3,2025-06-28 22:13:15.619,NaN,Media file,player #1:C:/Users/thoma/Desktop/BorisVidsObse...,0.0,589.517,622.862,29.0,subject,22.135,Facial Sniffing,Not defined,STATE,11.966,13.000,1.034,C:/Users/thoma/Desktop/BorisVidsObservation/RI...,347.0,377.0,NaN,NaN,NaN,NaN,RI1_3_6,Positive,3_6,NaN,NaN,NaN,NaN
2,RI1_s3_6_p5_3_nRB3,2025-06-28 22:13:15.619,NaN,Media file,player #1:C:/Users/thoma/Desktop/BorisVidsObse...,0.0,589.517,622.862,29.0,social_agent,23.313,Facial Sniffing,Not defined,STATE,12.069,12.862,0.793,C:/Users/thoma/Desktop/BorisVidsObservation/RI...,350.0,373.0,NaN,NaN,NaN,NaN,RI1_3_6,Positive,3_6,NaN,NaN,NaN,NaN
3,RI1_s3_6_p5_3_nRB3,2025-06-28 22:13:15.619,NaN,Media file,player #1:C:/Users/thoma/Desktop/BorisVidsObse...,0.0,589.517,622.862,29.0,subject,22.135,Body Sniffing,Not defined,STATE,13.276,13.448,0.172,C:/Users/thoma/Desktop/BorisVidsObservation/RI...,385.0,390.0,NaN,NaN,NaN,NaN,RI1_3_6,Positive,3_6,NaN,NaN,NaN,NaN
4,RI1_s3_6_p5_3_nRB3,2025-06-28 22:13:15.619,NaN,Media file,player #1:C:/Users/thoma/Desktop/BorisVidsObse...,0.0,589.517,622.862,29.0,subject,22.135,Anogenital Sniffing,Not defined,STATE,14.483,16.690,2.207,C:/Users/thoma/Desktop/BorisVidsObservation/RI...,420.0,484.0,NaN,NaN,NaN,NaN,RI1_3_6,Positive,3_6,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1014,RI2_s3_5_nRB3,2025-06-27 12:59:24.834,NaN,Media file,player #1:E:/Aim1/AIM1/Day1_new/RI2_s3_5_p5_4_...,0.0,208.866,602.033,30.0,social_agent,46.734,Body Sniffing,Not defined,STATE,80.733,82.100,1.367,E:/Aim1/AIM1/Day1_new/RI2_s3_5_p5_4_nRB3_20250...,2422.0,2463.0,NaN,NaN,NaN,NaN,RI2_3_5,Negative,3_5,NaN,NaN,NaN,NaN
1015,RI2_s3_5_nRB3,2025-06-27 12:59:24.834,NaN,Media file,player #1:E:/Aim1/AIM1/Day1_new/RI2_s3_5_p5_4_...,0.0,208.866,602.033,30.0,social_agent,46.734,Body Sniffing,Not defined,STATE,170.900,176.000,5.100,E:/Aim1/AIM1/Day1_new/RI2_s3_5_p5_4_nRB3_20250...,5127.0,5280.0,NaN,NaN,NaN,NaN,RI2_3_5,Negative,3_5,NaN,NaN,NaN,NaN
1016,RI2_s3_5_nRB3,2025-06-27 12:59:24.834,NaN,Media file,player #1:E:/Aim1/AIM1/Day1_new/RI2_s3_5_p5_4_...,0.0,208.866,602.033,30.0,social_agent,46.734,Facial Sniffing,Not defined,STATE,203.600,206.767,3.167,E:/Aim1/AIM1/Day1_new/RI2_s3_5_p5_4_nRB3_20250...,6108.0,6203.0,NaN,NaN,NaN,NaN,RI2_3_5,Negative,3_5,NaN,NaN,NaN,NaN
1017,RI2_s3_5_nRB3,2025-06-27 12:59:24.834,NaN,Media file,player #1:E:/Aim1/AIM1/Day1_new/RI2_s3_5_p5_4_...,0.0,208.866,602.033,30.0,social_agent,46.734,Facial Sniffing,Not defined,STATE,207.433,212.067,4.634,E:/Aim1/AIM1/Day1_new/RI2_s3_5_p5_4_nRB3_20250...,6223.0,6362.0,NaN,NaN,NaN,NaN,RI2_3_5,Negative,3_5,NaN,NaN,NaN,NaN


In [30]:
# --- Descriptive statistics: bout counts ---
bout_counts = (
    combined_bouts
    .groupby(["Subject_ID", "Condition", "Subject", "Behavior"])
    .size()
    .reset_index(name="Bout_Count")
)

print("✅ Bout counts ready")
bout_counts.head(20)


✅ Bout counts ready


,Subject_ID,Condition,Subject,Behavior,Bout_Count
0,1_1,Negative,social_agent,Body Sniffing,14
1,1_1,Negative,social_agent,Chasing,1
2,1_1,Negative,social_agent,Facial Sniffing,4
3,1_1,Negative,social_agent,Fighting,8
4,1_1,Negative,subject,Anogenital Sniffing,1
5,1_1,Negative,subject,Body Sniffing,2
6,1_1,Negative,subject,Posturing,7
7,1_1,Positive,social_agent,Body Sniffing,3
8,1_1,Positive,social_agent,Facial Sniffing,1
9,1_1,Positive,subject,Anogenital Sniffing,20


In [31]:
# --- Minimum bouts filter (example: at least 5 per condition) ---
X = 5  # you can change this
valid_subjects = (
    bout_counts
    .groupby(["Subject_ID", "Condition"])["Bout_Count"]
    .sum()
    .reset_index()
    .query("Bout_Count >= @X")["Subject_ID"]
    .unique()
)

filtered_bout_counts = bout_counts[bout_counts["Subject_ID"].isin(valid_subjects)]
print(f"✅ Filtered: {len(valid_subjects)} subjects met minimum {X} bouts")


✅ Filtered: 8 subjects met minimum 5 bouts


In [34]:
valid_subjects = filtered_bout_counts["Subject_ID"].unique()
print("Valid subjects:", valid_subjects)


Valid subjects: ['1_1' '1_2' '2_3' '2_4' '3_5' '3_6' '4_7' '4_8']


In [35]:
valid_bouts = bout_counts[bout_counts["Subject_ID"].isin(valid_subjects)]
valid_bouts.head()


,Subject_ID,Condition,Subject,Behavior,Bout_Count
0,1_1,Negative,social_agent,Body Sniffing,14
1,1_1,Negative,social_agent,Chasing,1
2,1_1,Negative,social_agent,Facial Sniffing,4
3,1_1,Negative,social_agent,Fighting,8
4,1_1,Negative,subject,Anogenital Sniffing,1


In [36]:
sniff_behaviors = ["Facial Sniffing", "Body Sniffing", "Anogenital Sniffing"]


In [37]:
subject_sniffs = bout_counts[
    (bout_counts["Subject"] == "subject") &
    (bout_counts["Behavior"].isin(sniff_behaviors))
]


In [40]:
min_bouts = 5

# Count total subject-initiated sniffs per subject (across all sniff types + conditions)
subject_totals = (
    subject_sniffs.groupby("Subject_ID")["Bout_Count"].sum().reset_index()
)

# Find which subjects meet the threshold
valid_subjects = subject_totals[subject_totals["Bout_Count"] >= min_bouts]["Subject_ID"].unique()

# Filter down
valid_subject_sniffs = subject_sniffs[subject_sniffs["Subject_ID"].isin(valid_subjects)]

print("✅ Subjects kept:", valid_subjects)
valid_subject_sniffs


✅ Subjects kept: ['1_1' '1_2' '2_3' '2_4' '3_6' '4_7' '4_8']


,Subject_ID,Condition,Subject,Behavior,Bout_Count
4,1_1,Negative,subject,Anogenital Sniffing,1
5,1_1,Negative,subject,Body Sniffing,2
9,1_1,Positive,subject,Anogenital Sniffing,20
10,1_1,Positive,subject,Body Sniffing,46
11,1_1,Positive,subject,Facial Sniffing,22
17,1_2,Negative,subject,Body Sniffing,1
18,1_2,Negative,subject,Facial Sniffing,2
23,1_2,Positive,subject,Anogenital Sniffing,16
24,1_2,Positive,subject,Body Sniffing,44
25,1_2,Positive,subject,Facial Sniffing,15
